# 31. Amounts that arrive as text

**Track:** pyspark | **Difficulty:** easy | **Tags:** data cleaning, casting, try_cast, regexp_replace, data types

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("031-amounts-stored-as-text")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "raw_payments": """
UEFSMRUEFeAPFYwITBX8ARUAEgAA8AcMcREBAAEBAHINCABzDQgAdA0IAHUNCAB2DQgAdw0IAHgNCAB5DQgAeg0IAHsNCAB8DQgA
fQ0IAH4NCAB/DQgAgA0IAIENCACCDQgAgw0IAIQNCACFDQgAhg0IAIcNCACIDQgAiQ0IAIoNCACLDQgAjA0IAI0NCACODQgAjw0I
AJANCACRDQgAkg0IAJMNCACUDQgAlQ0IAJYNCACXDQgAmA0IAJkNCACaDQgAmw0IAJwNCACdDQgAng0IAJ8NCACgDQgAoQ0IAKIN
CACjDQgApA0IAKUNCACmDQgApw0IAKgNCACpDQgAqg0IAKsNCACsDQgArQ0IAK4NCACvDQgAsA0IALENCACyDQgAsw0IALQNCAC1
DQgAtg0IALcNCAC4DQgAuQ0IALoNCAC7DQgAvA0IAL0NCAC+DQgAvw0IAMANCADBDQgAwg0IAMMNCADEDQgAxQ0IAMYNCADHDQgA
yA0IAMkNCADKDQgAyw0IAMwNCADNDQgAzg0IAM8NCADQDQgA0Q0IANINCADTDQgA1A0IANUNCADWDQgA1w0IANgNCADZDQgA2g0I
ANsNCADcDQgA3Q0IAN4NCADfDQgA4A0IAOENCADiDQgA4w0IAOQNCADlDQgA5g0IAOcNCADoDQgA6Q0IAOoNCADrDQgA7A0IPO0R
AQAAAAAA7hEBAAAAAAAVABXyARX4ASwV/AEVEBUGFQYcGAjuEQEAAAAAABgIcREBAAAAAAAWACgI7hEBAAAAAAAYCHERAQAAAAAA
EREAAAB58HgDAAAA/AEBByGAgGBAKBgOiIRiwWg4HpCIZEKpWC6YjGbD6Xg+oJBoRCqZTqiUasVquV6wmGxGq9luuJxux+v5fsCg
cEgsGo/IpHLJbDqf0Kh0Sq1ar9isdsvter/gsHhMLpvP6LR6zW673/C4fE6v2+/4vH7P7wMAFQQVvhkV8BNMFfoBFQASAADfDDwM
AAAA4oK5NjksNTIyLjUwARDwPDIsMTIwLDU3Ny4wMA4AAABJTlIgNDE2NTA5NC41MAYAAAAxMDMuNTADAAAAICAgCAAAADQsODA0
Ljk5CgAJLzw4NzAuNTAJAAAANjc4NDUxDTgkUnMgOTA2AAAAAAEpFOKCuSA4OAEpRA0AAAAgIDE4MDc4OTguNTAgCwURDDU0NTIF
DwFJHDU2LDI5Ni4wES0QNTIzNDcBkSwgBwAAAC00NzguOTkBnQA1BQoBQigtMTg5NzE0OC4xNgHYKOKCuTEwLDU0NS44BcYINTky
DeIBiiQxNDM5MzE5LjUwEZwQMjE5LjAFKgg5NTYNzyBScyAzMDk4MDUBahTigrk4NDEBCgwxNjAuGf4UOTkzLjk5AXMhOxg2MDYz
OC44BRAsUnMuIDMxNTEyLjk5AdUUMjYzODQ1DQ0cICA3NzkuOTkF7wwgIDM2AdIMIA8AAAV2GDYsNDEsMzMhZhAFAAAANAGSBAwA
Jb4INzMsAVMEMzYJ9SAzNTAzNTcuNTAhGTRScyA1MjA3AwAAAE4vQQGsBDY5ATsBfiBScyA4MjUuNDQBFxA3NzEuMAUzFDIzMzgy
OQXYBDI3AUghnwG6IDc2MTAyMy4zNxE9ADkBogGhLOKCuTMsNjkzLDk2M01GGCAgNDcyMzQBVQAgAWAEMTQhbgGICDkyMgGpROKC
uTksOTA3LjE1BAAAADAuMAXQHDIsMzA5LDQ2QX4ECgAJ/BQ3NjA3MzdRjBQxMTk3ODgZxQQuIEVFAR80dHdlbHZlIGh1bmRyZWQt
HxAgNjMyMUE6AXYUICA5OTQzAW0AIAleDC4gODgBxwGEDE5VTEwhNQgtODIBMAHnJcIMMy4wNgG6CFJzLg3uHDEsOTgwLDkxAd8B
SCQyNjM3Njk3LjgzAYcBNRQzNDQ2OThBUAH+FDQ1NC44MwEqQWwENjIBOCHHDDMxLjYlfwQyMEEHAT0BIRQzNjI2OTVBfwEzIXwA
OQUzFAIAAAAtLWFyHFJzIDkyMjcyAZcIOTE1AbYUMjYxMTM3DTIcIDIwMzAzNDIBfQQxNiHGAUoBXBA2MTQuNSUMCDEyOQFjDCBV
U0QFJwWDITUFcQg1MjkhhAGTAEmBAgw1NTU0RVcBIkH7DDYsODFR6UFLDDAsMDYB8CE3QREMMiw4NQV2IC00MzMzNzcuOVEsIDQs
NTI2LDQ0OC30AXAEMjIJjgGnJDE0NTUxNzEuOTkB2AQtOIVBIS8QNTQ5NjQBYBAtMjcwNmFGAbkINTE1TbsYLTMyNC41OAEiIDk1
LDI3Ni45OSEiCGFiYwHIAf8gMTE0NTQzMi4zBSYBEhQ0NTAuNQshWhAxMTcwNmX2AAFBkgGLJDQ3Mjc4MzAuNTQBgKErADMtACAx
LDUzNSw2MTUBbgFIADYh1QENCCAgMSG8BDcgAZ0UMTIuNS4zLbMMNzI4MWW/EDEgMjk5DRUQODk3ODEBpgjigrkBcyEOFDU2Ni41
ME1tHDMsNTA4LDk0QbYNfQwgNjA1ZX4VMQA5QUABHhw0MCwwMCw5OCH0AY4BtAA4ATsBDQw1Niw1BeEBmwA0JSABNChScyAxMCww
OTkuOGWcYb0UMTA1LjUzhYA44oK5Miw2MDMsNTM2LjY0AXQB8QQ1MCGjAVcQNjIsMzQhzjgLAAAAICAyOTA0My41MCAVABX6ARWA
AiwV/AEVEBUGFQYcNgIoC+KCuTksOTA3LjE1GAAREQAAAH3wfAcAAACoAQED/kQBByGAgGBAKBgOiIRiwWg4HpCIZEKpWC6YjGbD
6Xg+oJBoRCqZTqiUasVquV6wmGxGq9luuJxux+v5fsCgcEgsGo/IpHLJbDqf0Kh0Sq1ar9isdsvter/gsHhMLpvP6LR6zW673/C4
fE6v2+/4vH7PBwAAFQQZPDUAGAZzY2hlbWEVBAAVBCUCGApwYXltZW50X2lkABUMJQIYC2Ftb3VudF90ZXh0JQBMHAAAABb8ARkc
GSwmABwVBBk1AAYQGRgKcGF5bWVudF9pZBUCFvwBFvgSFqoLJrYIJggcGAjuEQEAAAAAABgIcREBAAAAAAAWACgI7hEBAAAAAAAY
CHERAQAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAPwBAAAAJgAcFQwZNQAGEBkYC2Ftb3VudF90ZXh0FQIW/AEWrBwW5BYm
xB8msgscNgIoC+KCuTksOTA3LjE1GAAREQAZLBUEFQAVAgAVABUQFQIAPBbWERkGGSYC+gEAAAAWpC8W/AEmCBaOIgAZHBgMQVJS
T1c6c2NoZW1hGIACLy8vLy83Z0FBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFB
QUJBQUFBQUlBQUFCSUFBQUFCQUFBQU5ELy8vOEFBQUVGRUFBQUFDQUFBQUFFQUFBQUFBQUFBQXNBQUFCaGJXOTFiblJmZEdWNGRB
QUVBQVFBQkFBQUFCQUFGQUFJQUFZQUJ3QU1BQUFBRUFBUUFBQUFBQUFCQWhBQUFBQWtBQUFBQkFBQUFBQUFBQUFLQUFBQWNHRjVi
V1Z1ZEY5cFpBQUFDQUFNQUFnQUJ3QUlBQUFBQUFBQUFVQUFBQUFBQUFBQQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUu
MC4xGSwcAAAcAAAAWAIAAFBBUjE=
""",
}

_data_dir = pathlib.Path("data") / "031-amounts-stored-as-text"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

raw_payments = spark.read.parquet((_data_dir / "raw_payments.parquet").resolve().as_posix())
raw_payments.createOrReplaceTempView("raw_payments")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'amount': 'c7eaba6ae91be685a18d8d5ce7310e3597b8db080217a1eeccf1ee1d2b974943',
                         'is_valid': 'ac29efa791cb98e6c8df4f6582b0f7c58d912649b1cd4dbeb1849cf23e0aee1e',
                         'payment_id': 'a695ad125407309132ddcf582da0835fe23cc18ceb0aba1e0c79a5cc84ee8858'},
 'columns': [['amount', 'decimal(12,2)'], ['is_valid', 'boolean'], ['payment_id', 'bigint']],
 'fingerprint': '2a4b6853cc44b76cede085899bff250e669cbb61ad6ce78a3a7b10438078cae5',
 'order_matters': False,
 'row_count': 126}

check = make_check(EXPECTED)

## The situation

A partner sends payments in a spreadsheet export. The amount column was typed by
people, so it is text: sometimes with a rupee sign, sometimes with thousands
separators, sometimes "N/A".

The load job casts the column to a decimal. In the old cluster the bad values quietly
became NULL and nobody knew how many there were. After the upgrade to Spark 4 the job
does not finish at all: one bad value stops it.

## Input tables

### `raw_payments`

One row per payment.

| Column | Type | Description |
|---|---|---|
| `payment_id` | BIGINT | Unique per payment. |
| `amount_text` | STRING | The amount as it was typed. Can be NULL or empty. |

Look at the variety first: `raw_payments.select("amount_text").distinct().show(200, truncate=False)`.

## Requirement

Return every payment with its amount as a number.

To read an amount:

1. Remove the rupee markers `₹`, `Rs.`, `Rs` and `INR`, and every comma. Commas
   are thousands separators, in western (`1,234,567`) or Indian (`12,34,567`)
   grouping.
2. Remove spaces at the start and the end.
3. If what is left is a plain number, such as `1299`, `1299.50` or `-200.00`, that
   is the amount. A minus sign means a refund and is valid. So is `0`.
4. Anything else is not an amount: `N/A`, an empty value, `12.5.3`, `1299.00 USD`.

- `amount` is the number, and NULL when the text is not an amount.
- `is_valid` is true when `amount` is not NULL and false otherwise.
- Every payment appears, valid or not. The job must not fail on bad values.

## Expected output

One row per payment.

| Column | Type | Description |
|---|---|---|
| `payment_id` | BIGINT | |
| `amount` | DECIMAL(12,2) | NULL if the text is not an amount. |
| `is_valid` | BOOLEAN | |

In [ ]:
raw_payments.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `raw_payments`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)